# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "alumno01", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_tfernandezaddoumie/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = None

from pyspark.sql import functions as F

# leo events_json_v2
challenge = spark.read.json(challenge_path)

# schema del events_json_v2
challenge.printSchema()

# leo lo que ya tenemos en bronze del 01
historico = spark.table(f"{config.catalog}.{config.schema}.bronze_events")

# columnas que no estaban en bronze
print("Columnas nuevas:", set(challenge.columns) - set(historico.columns))

# event_type nuevo
tipos_nuevos = (challenge.select("event_type").distinct()
                .subtract(historico.select("event_type").distinct()))
display(tipos_nuevos)

# filas y enventos distintos en events_json_v2
display(challenge.agg(
    F.count("*").alias("filas"),
    F.countDistinct("event_id").alias("ids_distintos"),
))

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)

Columnas nuevas: {'app_version'}


event_type
refund


filas,ids_distintos
10001,10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# TODO: unir por nombre admitiendo columnas faltantes, deduplicar y escribir target_table.

# historico bronze_events
historico = spark.table(source_table)

# nuevo, de challenge celda anterior y le agrego trazabilidad (source, source_file, ingested_at)
nuevo = (challenge
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(challenge_path))
    .withColumn("_ingested_at", F.current_timestamp()))

# uno por nombre de columna, completando con NULL lo que falte (lo que falta va a ser el app_version en historico)
unido = historico.unionByName(nuevo, allowMissingColumns=True)

# dedup una fila por event_id
bronze_v2 = unido.dropDuplicates(["event_id"])

# guardoo como Delta, reemplazando si ya existe
(bronze_v2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(target_table))

print(f"{target_table}: {spark.table(target_table).count():,} filas")

workspace.bigdata_tfernandezaddoumie.bronze_events_v2: 210,000 filas


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = result.count() 
distinct_ids = result.select("event_id").distinct().count()
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.

Detectar una evolución de esquema es darse cuenta de que algo cambió. En este caso fue ver que apareció la columna app_version y un tipo de evento nuevo, refund.

Aceptarla técnicamente es lo que hicimos con bronze_events_v2. Es decir logramos que el pipeline absorba el cambio sin romperse. En Bronze guardamos todo lo que llega y todavia no me pregunto si nos va a servir o no.

 Un cambio puede ser técnicamente aceptable y aun así no ser válido para el negocio. Por ejemplo, este evento de tipo refund, es un evento? no sería una transacción que reversa una compra? Tal vez nos sirve para entender el ingreso efectivo de dinero por ventas pero no para analizar fraude. Esa decisión se debería aplicar en Silver o Gold.